# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

## Rules

This notebook is checked automatically by **ThetaGrader (TG)**.

- Keep every `### Task N` heading intact -- TG uses it as a marker to locate tasks. Do not remove, rename, renumber, or duplicate these headings.
- Write your solution in the cell(s) reserved for it, directly below the corresponding task statement. Do not move a solution to a different place in the notebook or merge several tasks' solutions together.
- If a task asks for a written conclusion, report, or answer to a theoretical question, put it in a markdown cell directly under that task, clearly separated from code.
- Before submitting your final solution, make sure the `Status` field in the Bot-check cell below is set to `final`.

## AI usage disclosure

I used ChatGPT to understand how to approach the tasks, optimize training and evaluation, and help write some code snippets. =)

In [2]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Data Preparation

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B / 70.8MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/7.83M [00:00<?, ?B/s]

...

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [4]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  433MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [6]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [7]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [8]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [9]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

### Task 1 (1 point):

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)

In [10]:
from tqdm.auto import tqdm

correct = 0
total = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    collate_fn=transformers.default_data_collator,
    num_workers=2,
)

with torch.no_grad():
    progress = tqdm(val_loader, desc="Валидация")

    for batch in progress:
        batch = {key: value.to(device) for key, value in batch.items()}

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch["token_type_ids"],
        )
        predictions = outputs.logits.argmax(dim=1)

        correct += (predictions == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

        progress.set_postfix(accuracy=f"{correct / total:.4f}")

accuracy = correct / total
print(f"Validation accuracy: {accuracy:.4f}")

Валидация:   0%|          | 0/1264 [00:00<?, ?it/s]

Validation accuracy: 0.9084


In [ ]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

### Task 2 (4 points):
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [11]:
# Если зависимости уже установлены, повторная установка ничего не изменит
%pip -q install accelerate sentencepiece protobuf

import gc
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("Включи GPU: Среда выполнения → Сменить среду выполнения → GPU")

# Освобождаем GPU от BERT из Task 1
if "model" in globals():
    model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

# 1. Исходные тексты, а не токены BERT
qqp_raw = load_dataset("nyu-mll/glue", "qqp")

# Убираем записи без одного из вопросов
qqp_raw = qqp_raw.filter(
    lambda example: bool(example["question1"] and example["question2"])
)

# 2. Токенизатор именно выбранной модели
MODEL_NAME = "microsoft/deberta-v3-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
)

def tokenize_pairs(batch):
    return deberta_tokenizer(
        batch["question1"],
        batch["question2"],
        truncation=True,
        max_length=128,
        # Padding сделает collator отдельно для каждого батча
    )

train_data = qqp_raw["train"].map(
    tokenize_pairs,
    batched=True,
    remove_columns=["question1", "question2", "idx"],
)

val_data = qqp_raw["validation"].map(
    tokenize_pairs,
    batched=True,
    remove_columns=["question1", "question2", "idx"],
)

# Trainer ожидает столбец labels
train_data = train_data.rename_column("label", "labels")
val_data = val_data.rename_column("label", "labels")

# 3. Предобученный body + новая head для двух классов
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32,
)

deberta_model.gradient_checkpointing_disable()

# Явно разрешаем обучение всех параметров
for parameter in deberta_model.parameters():
    parameter.requires_grad_(True)

# Экономим память GPU
deberta_model.gradient_checkpointing_disable()

# 4. Метрика
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": float(np.mean(predictions == labels))}

# Сохраняем только номера классов, а не все logits
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics_from_classes(eval_pred):
    predictions, labels = eval_pred
    return {"accuracy": float(np.mean(predictions == labels))}

# 5. Настройки обучения
training_args = TrainingArguments(
    output_dir="/content/deberta_qqp_checkpoints",
    num_train_epochs=2,
    learning_rate=2e-5,

    # per_device_train_batch_size=8,
    # gradient_accumulation_steps=4,  # эффективный батч: 8 × 4 = 32
    # per_device_eval_batch_size=16,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,  # эффективный батч остаётся 32
    per_device_eval_batch_size=8,

    weight_decay=0.01,
    warmup_steps=190,
    max_grad_norm=1.0,
    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1,

    logging_steps=100,
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
    gradient_checkpointing=False,
)

train_small = train_data.shuffle(seed=42).select(
    range(min(30_000, len(train_data)))
)

trainer = Trainer(
    model=deberta_model,
    args=training_args,
    # train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorWithPadding(
        tokenizer=deberta_tokenizer,
        pad_to_multiple_of=8,
    ),
    compute_metrics=compute_metrics_from_classes,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    train_dataset=train_small,
)

# 6. Обучение — прогресс отображается автоматически
trainer.train()

# Здесь уже используется лучшая модель по validation accuracy
metrics = trainer.evaluate()
deberta_accuracy = metrics["eval_accuracy"]

print(f"DeBERTa validation accuracy: {deberta_accuracy:.4f}")

# accuracy из Task 1 не перезаписываем
if "accuracy" in globals():
    print(f"BERT validation accuracy:    {accuracy:.4f}")
    print(f"Разница: {deberta_accuracy - accuracy:+.4f}")

# 7. Сохраняем веса и токенизатор в среде Colab
trainer.save_model("/content/deberta_qqp_final")
deberta_tokenizer.save_pretrained("/content/deberta_qqp_final")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

qqp/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 33.6MB            

qqp/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

qqp/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.73MB            

qqp/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
1.692877,0.288934,2,0.883601


DeBERTa validation accuracy: 0.8836
BERT validation accuracy:    0.9084
Разница: -0.0248


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/deberta_qqp_final/tokenizer_config.json',
 '/content/deberta_qqp_final/tokenizer.json')

Выбрана Option A: fine-tuning microsoft/deberta-v3-base. Обучались все параметры модели на 30 000 парах QQP в течение двух эпох. Learning rate — 2e-5, эффективный batch size — 32, максимальная длина — 128. Validation accuracy составила 0.8836, у исходного BERT — 0.9084. DeBERTa уступила на 2.48 процентного пункта. Возможная причина — ограниченная обучающая выборка и короткое обучение.

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

### Task 3 (1 point):
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [12]:
import pandas as pd

# После обучения Trainer загрузил лучшую модель
duplicate_model = trainer.model
device = next(duplicate_model.parameters()).device
duplicate_model.eval()

# Собираем уникальные вопросы из обоих столбцов train
train_questions = list(dict.fromkeys(
    question.strip()
    for column in ["question1", "question2"]
    for question in qqp_raw["train"][column]
    if question and question.strip()
))

print("Уникальных вопросов:", len(train_questions))


@torch.inference_mode()
def find_duplicates(question, top_k=5, batch_size=32):
    question = question.strip()
    if not question:
        raise ValueError("Введи непустой вопрос")

    # Исключаем буквальное совпадение с запросом
    candidates = [
        text for text in train_questions
        if text.casefold() != question.casefold()
    ]

    best_scores = np.empty(0, dtype=np.float32)
    best_questions = []

    for start in tqdm(
        range(0, len(candidates), batch_size),
        desc="Поиск дубликатов",
    ):
        texts = candidates[start:start + batch_size]

        inputs = deberta_tokenizer(
            [question] * len(texts),
            texts,
            padding=True,
            truncation=True,
            max_length=128,  # как при обучении
            return_tensors="pt",
        ).to(device)

        logits = duplicate_model(**inputs).logits
        scores = logits.float().softmax(dim=-1)[:, 1].cpu().numpy()

        # Храним только top_k, а не все результаты
        merged_scores = np.concatenate([best_scores, scores])
        merged_questions = best_questions + texts
        indices = np.argsort(merged_scores)[-top_k:][::-1]

        best_scores = merged_scores[indices]
        best_questions = [merged_questions[i] for i in indices]

    return pd.DataFrame({
        "question": best_questions,
        "duplicate_score": best_scores,
    })


# Выбираем пять разных запросов с положительной разметкой
examples = []
seen = set()

for row in qqp_raw["validation"]:
    query = row["question1"]

    if row["label"] == 1 and query.casefold() not in seen:
        examples.append(row)
        seen.add(query.casefold())

    if len(examples) == 5:
        break

search_results = []

for i, row in enumerate(examples, start=1):
    print(f"\nПример {i}")
    print("Запрос:", row["question1"])
    print("Дубликат по разметке validation:", row["question2"])
    print(
        "Этот дубликат есть среди кандидатов train:",
        row["question2"].strip() in train_questions,
    )

    result = find_duplicates(row["question1"])
    search_results.append(result)
    display(result)

Уникальных вопросов: 493854

Пример 1
Запрос: Is there a reason why we should travel alone?
Дубликат по разметке validation: What are some reasons to travel alone?
Этот дубликат есть среди кандидатов train: True


Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

,question,duplicate_score
0,Why should I travel alone?,0.960510
1,What are some reasons to travel alone?,0.953880
2,If the Indian government has decided to demone...,0.928603
3,Should I travel alone?,0.925836
4,Why does 500 and 1000 Rs notes banned by GOI a...,0.906612



Пример 2
Запрос: I am 25 year old guy and never had a girlfriend. Is this weird?
Дубликат по разметке validation: I am 25 years old. I have never had a girlfriend. Is something wrong with me?
Этот дубликат есть среди кандидатов train: True


Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

...

Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

,question,duplicate_score
0,What should I do in a weekend layover in Hong ...,0.943504
1,What should I do in Hong Kong?,0.935817
2,Why does 500 and 1000 Rs notes banned by GOI a...,0.902610
3,If the Indian government has decided to demone...,0.871037
4,I'm planing to stop at Hong kong for a day. An...,0.864081



Пример 5
Запрос: Which is the best gaming laptop under 40k?
Дубликат по разметке validation: Which is the best gaming laptop under 40,000 rs?
Этот дубликат есть среди кандидатов train: True


Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

,question,duplicate_score
0,"Which is the best gaming laptop under 40,000 rs?",0.981022
1,What is best gaming laptop in the range Rs 40-...,0.945197
2,Can you suggest a good laptop within ₹40k?,0.925701
3,Which laptop is best under 40000 INR?,0.924959
4,What is the best laptop to buy up to Rs 40000 ...,0.922273


Функция протестирована на пяти запросах. Она находит правильные переформулировки: например, для путешествий в одиночку и игрового ноутбука до 40 000 рупий. Однако в нескольких выдачах встречаются явно нерелевантные вопросы про отмену индийских банкнот. Следовательно, высокая оценка класса «дубликат» не всегда означает смысловое совпадение. Для запроса про Quora правильная переформулировка находится лишь на третьем месте. Модель частично решает задачу поиска, но допускает уверенные ложноположительные ответы.

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

### Task 4 (Bonus, 0.5 point):
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [14]:
# 3. Сравниваем обе функции на одном запросе
query = examples[0]["question1"]
duplicate_model.eval()

def synchronize_gpu():
    model_device = next(duplicate_model.parameters()).device
    if model_device.type == "cuda":
        torch.cuda.synchronize(model_device)

def timed_search(function, question):
    synchronize_gpu()
    started = time.perf_counter()

    result = function(question, batch_size=32)

    synchronize_gpu()
    elapsed = time.perf_counter() - started
    return result, elapsed

# Прогреваем модель перед измерениями
_ = find_duplicates_fast(query, candidate_count=32)

print("Запрос:", query)
print("Запускаем полный поиск — он может занять несколько минут...")
slow_result, slow_seconds = timed_search(find_duplicates, query)

print("Запускаем быстрый поиск...")
fast_result, fast_seconds = timed_search(find_duplicates_fast, query)

print("\nПолный поиск:")
display(slow_result)

print("Быстрый поиск:")
display(fast_result)

overlap = len(
    set(slow_result["question"]) & set(fast_result["question"])
)

display(pd.DataFrame([{
    "Полный поиск, сек": slow_seconds,
    "Быстрый поиск, сек": fast_seconds,
    "Ускорение, раз": slow_seconds / fast_seconds,
    "Совпадений в top-5": overlap,
    "Построение индекса, сек": index_seconds,
}]).round(2))

Запрос: Is there a reason why we should travel alone?
Запускаем полный поиск — он может занять несколько минут...


Поиск дубликатов:   0%|          | 0/15433 [00:00<?, ?it/s]

Запускаем быстрый поиск...

Полный поиск:


,question,duplicate_score
0,Why should I travel alone?,0.960510
1,What are some reasons to travel alone?,0.953880
2,If the Indian government has decided to demone...,0.928603
3,Should I travel alone?,0.925836
4,Why does 500 and 1000 Rs notes banned by GOI a...,0.906612


Быстрый поиск:


,question,duplicate_score
0,Why should I travel alone?,0.960510
1,What are some reasons to travel alone?,0.953837
2,Should I travel alone?,0.925903
3,Why do people like to travel alone?,0.752922
4,Is it safe to travel alone?,0.698072


,"Полный поиск, сек","Быстрый поиск, сек","Ускорение, раз",Совпадений в top-5,"Построение индекса, сек"
0,718.97,0.44,1638.36,3,14.14


### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

### Task 5 (Bonus, 1.5 points):
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [17]:
def find_duplicates_glove(question, top_k=5):
    question = question.strip()

    if not question:
        raise ValueError("Введи непустой вопрос")
    if top_k < 1:
        raise ValueError("top_k должен быть положительным")

    vector = glove_question_vector(question)

    if not np.any(vector):
        raise ValueError("В GloVe нет известных слов для этого запроса")

    scores = glove_question_matrix @ vector

    excluded = (
        (glove_questions_lower == question.casefold())
        | ~glove_valid_vectors
    )
    scores[excluded] = -np.inf

    count = min(top_k, int((~excluded).sum()))
    if count == 0:
        return pd.DataFrame(columns=["question", "cosine_similarity"])

    indices = np.argpartition(scores, len(scores) - count)[-count:]
    indices = indices[np.argsort(scores[indices])[::-1]]

    return pd.DataFrame({
        "question": [train_questions[i] for i in indices],
        "cosine_similarity": scores[indices],
    })

In [18]:
%pip -q install -U gensim

import time
import numpy as np
import pandas as pd
import gensim.downloader as api

from tqdm.auto import tqdm

# Загрузка измеряется отдельно от построения индекса
started = time.perf_counter()
glove = api.load("glove-wiki-gigaword-100")
glove_load_seconds = time.perf_counter() - started

print(f"Загрузка GloVe: {glove_load_seconds:.2f} сек.")

# Используем IDF из Task 4.
# Для усреднения нужны только отдельные слова, без биграмм.
idf_weights = dict(zip(
    tfidf_vectorizer.get_feature_names_out(),
    tfidf_vectorizer.idf_,
))
tokenize_words = tfidf_vectorizer.build_tokenizer()
preprocess_text = tfidf_vectorizer.build_preprocessor()


def glove_question_vector(text):
    words = tokenize_words(preprocess_text(text))

    vector = np.zeros(glove.vector_size, dtype=np.float32)
    total_weight = 0.0

    for word in words:
        if word in glove:
            weight = idf_weights.get(word, 1.0)
            vector += weight * glove[word]
            total_weight += weight

    if total_weight > 0:
        vector /= total_weight

    norm = np.linalg.norm(vector)
    if norm > 0:
        vector /= norm

    return vector


started = time.perf_counter()

# Около 198 МБ для 494 тысяч вопросов × 100 float32
glove_question_matrix = np.empty(
    (len(train_questions), glove.vector_size),
    dtype=np.float32,
)

for i, text in enumerate(tqdm(train_questions, desc="Индекс GloVe")):
    glove_question_matrix[i] = glove_question_vector(text)

# Подготовка для исключения точных совпадений
glove_questions_lower = np.array([
    text.casefold() for text in train_questions
])
glove_valid_vectors = np.linalg.norm(glove_question_matrix, axis=1) > 0

glove_index_seconds = time.perf_counter() - started

print(f"Построение индекса: {glove_index_seconds:.2f} сек.")
print(f"Вопросов без известных слов: {(~glove_valid_vectors).sum()}")

# query, slow_result, fast_result и их время — из Task 4
print("Запрос для сравнения:", query)

_ = find_duplicates_glove(query)  # прогрев

glove_times = []
for _ in range(5):
    started = time.perf_counter()
    glove_result = find_duplicates_glove(query)
    glove_times.append(time.perf_counter() - started)

glove_seconds = float(np.median(glove_times))

speed_comparison = pd.DataFrame([
    {
        "Метод": "Полный поиск DeBERTa",
        "Поиск, сек": slow_seconds,
        "Ускорение относительно полного": 1.0,
    },
    {
        "Метод": "TF-IDF → 200 → DeBERTa",
        "Поиск, сек": fast_seconds,
        "Ускорение относительно полного": slow_seconds / fast_seconds,
    },
    {
        "Метод": "GloVe + TF-IDF-веса",
        "Поиск, сек": glove_seconds,
        "Ускорение относительно полного": slow_seconds / glove_seconds,
    },
])

display(speed_comparison.round(3))

print(f"Индекс TF-IDF: {index_seconds:.2f} сек.")
print(f"Загрузка GloVe: {glove_load_seconds:.2f} сек.")
print(f"Индекс вопросов GloVe: {glove_index_seconds:.2f} сек.")

for name, result in [
    ("Полный поиск DeBERTa", slow_result),
    ("TF-IDF + DeBERTa", fast_result),
    ("GloVe", glove_result),
]:
    print("\n", name)
    display(result)

Загрузка GloVe: 83.10 сек.


Индекс GloVe:   0%|          | 0/493854 [00:00<?, ?it/s]

Построение индекса: 52.41 сек.
Вопросов без известных слов: 16
Запрос для сравнения: Is there a reason why we should travel alone?


,Метод,"Поиск, сек",Ускорение относительно полного
0,Полный поиск DeBERTa,718.967,1.000
1,TF-IDF → 200 → DeBERTa,0.439,1638.357
2,GloVe + TF-IDF-веса,0.040,18011.770


Индекс TF-IDF: 14.14 сек.
Загрузка GloVe: 83.10 сек.
Индекс вопросов GloVe: 52.41 сек.

 Полный поиск DeBERTa


,question,duplicate_score
0,Why should I travel alone?,0.960510
1,What are some reasons to travel alone?,0.953880
2,If the Indian government has decided to demone...,0.928603
3,Should I travel alone?,0.925836
4,Why does 500 and 1000 Rs notes banned by GOI a...,0.906612



 TF-IDF + DeBERTa


,question,duplicate_score
0,Why should I travel alone?,0.960510
1,What are some reasons to travel alone?,0.953837
2,Should I travel alone?,0.925903
3,Why do people like to travel alone?,0.752922
4,Is it safe to travel alone?,0.698072



 GloVe


,question,cosine_similarity
0,"""If """"time"""" does not exists, why do people st...",0.973789
1,Do you believe in time travel? Why?,0.973242
2,"Are we here for a reason, and, if so, what is ...",0.970999
3,I really want to travel the world but is it ac...,0.970962
4,What is it like to travel alone? How does it f...,0.970731


In [19]:
glove_results = []

for row in examples:
    print("\nЗапрос:", row["question1"])
    result = find_duplicates_glove(row["question1"])
    glove_results.append(result)
    display(result)

quality_rows = []

for name, result in [
    ("Полный поиск DeBERTa", slow_result),
    ("TF-IDF → 200 → DeBERTa", fast_result),
    ("GloVe + TF-IDF-веса", glove_result),
]:
    print("\nЗапрос:", query)
    print(name)
    display(result)

    while True:
        answer = input(
            f"Введи {len(result)} меток 0/1 через пробел "
            "в порядке строк таблицы: "
        ).split()

        if (
            len(answer) == len(result)
            and all(value in {"0", "1"} for value in answer)
        ):
            break

        print("Проверь количество меток и используй только 0 или 1.")

    labels = [int(value) for value in answer]

    quality_rows.append({
        "Метод": name,
        "Precision@5": np.mean(labels),
    })

quality_comparison = pd.DataFrame(quality_rows)
final_comparison = speed_comparison.merge(quality_comparison, on="Метод")
display(final_comparison.round(3))


Запрос: Is there a reason why we should travel alone?


,question,cosine_similarity
0,"""If """"time"""" does not exists, why do people st...",0.973789
1,Do you believe in time travel? Why?,0.973242
2,"Are we here for a reason, and, if so, what is ...",0.970999
3,I really want to travel the world but is it ac...,0.970962
4,What is it like to travel alone? How does it f...,0.970731



Запрос: I am 25 year old guy and never had a girlfriend. Is this weird?


,question,cosine_similarity
0,I am 25 years old. I have never had a girlfrie...,0.978839
1,I just got my first girlfriend. What should I ...,0.975790
2,I am 20 year old guy how can I get my first love?,0.974968
3,"I'm a 21 year old guy, I really like a girl wh...",0.972693
4,I am a 20-year-old guy. I don't have a girlfri...,0.972453



Запрос: "What does a good answer on Quora look like? What does it mean to ""be helpful""?"


...

,question,cosine_similarity
0,Which is the best gaming laptop under 50k?,0.950793
1,Which is best laptop under 40k?,0.942906
2,which is the best gaming laptop under 50000?,0.938762
3,Which one is best laptop under 40k?,0.938138
4,Which is the best gaming laptop under $500?,0.935573



Запрос: Is there a reason why we should travel alone?
Полный поиск DeBERTa


,question,duplicate_score
0,Why should I travel alone?,0.960510
1,What are some reasons to travel alone?,0.953880
2,If the Indian government has decided to demone...,0.928603
3,Should I travel alone?,0.925836
4,Why does 500 and 1000 Rs notes banned by GOI a...,0.906612


Введи 5 меток 0/1 через пробел в порядке строк таблицы: 11000
Проверь количество меток и используй только 0 или 1.
Введи 5 меток 0/1 через пробел в порядке строк таблицы: 11000
Проверь количество меток и используй только 0 или 1.
Введи 5 меток 0/1 через пробел в порядке строк таблицы: 11000
Проверь количество меток и используй только 0 или 1.
Введи 5 меток 0/1 через пробел в порядке строк таблицы: 11000
Проверь количество меток и используй только 0 или 1.
Введи 5 меток 0/1 через пробел в порядке строк таблицы: 11000
Проверь количество меток и используй только 0 или 1.
Введи 5 меток 0/1 через пробел в порядке строк таблицы: 11000
Проверь количество меток и используй только 0 или 1.
Введи 5 меток 0/1 через пробел в порядке строк таблицы: 
Проверь количество меток и используй только 0 или 1.
Введи 5 меток 0/1 через пробел в порядке строк таблицы: 
Проверь количество меток и используй только 0 или 1.
Введи 5 меток 0/1 через пробел в порядке строк таблицы: 
Проверь количество меток и исполь

KeyboardInterrupt: Interrupted by user